# MIDI parsing and df creation

## Import libraries

In [1]:
import os
from mido import MidiFile
import mido
import pandas as pd
import numpy as np

## Load MIDI file

In [2]:
def load_midi():
    
    # --- 1. mids/ ディレクトリを定義・作成 ---
    mids_dir = "mids"
    os.makedirs(mids_dir, exist_ok=True)
    # --- 修正完了 ---
    
    global midifile_path

    # --- 2. ファイル名のみを尋ねるように変更 ---
    mid_basename = input(f"Please insert the MIDI file name (must be in '{mids_dir}' folder) >> ")
    
    # --- 3. フルパスを自動で構築 ---
    midifile_path = os.path.join(mids_dir, mid_basename.strip()) # .strip() で前後の空白を削除

    def path_check(path):
        if os.path.exists(path):
            accept_ext = [".mid", ".midi", ".kar"]
            split_path = os.path.splitext(path)
            if split_path[1].lower() not in accept_ext: # .lower() で大文字の拡張子も許容
                print("Invalid file format. Please insert the path to a standard MIDI file: ")
                return False
            else:
                return True
        else:
            # --- 4. エラーメッセージを分かりやすく変更 ---
            print(f"Invalid path. File not found at: {path}")
            print(f"Please make sure '{mid_basename.strip()}' is inside the '{mids_dir}' folder.")
            return False

    # --- 5. ループ内の処理もファイル名入力 -> パス構築に変更 ---
    while not path_check(midifile_path):
        mid_basename = input(f"Please insert the MIDI file name (in '{mids_dir}' folder) >> ")
        midifile_path = os.path.join(mids_dir, mid_basename.strip())
    # --- 修正完了 ---

    print("Valid path.")
    print(f"Loading file: {midifile_path}")
    
load_midi()

Please insert the MIDI file name (must be in 'mids' folder) >>  complex_solo.mid


Valid path.
Loading file: mids/complex_solo.mid


## MIDI file df creation

In [11]:
import os
from mido import MidiFile
import mido
import pandas as pd
import numpy as np

# Note names definition (Global)
notes_names = ["C-1", "Cs-1", "D-1", "Ds-1", "E-1", "F-1", "Fs-1", "G-1", "Gs-1", "A-1", "As-1", "B-1", "C0", "Cs0", "D0", "Ds0", "E0", "F0", "Fs0", "G0", "Gs0", "A0", "As0", "B0", "C1", "Cs1", "D1", "Ds1", "E1", "F1", "Fs1", "G1", "Gs1", "A1", "As1", "B1", "C2", "Cs2", "D2", "Ds2", "E2", "F2", "Fs2", "G2", "Gs2", "A2", "As2", "B2", "C3", "Cs3", "D3", "Ds3", "E3", "F3", "Fs3", "G3", "Gs3", "A3", "As3", "B3", "C4", "Cs4", "D4", "Ds4", "E4", "F4", "Fs4", "G4", "Gs4", "A4", "As4", "B4", "C5", "Cs5", "D5", "Ds5", "E5", "F5", "Fs5", "G5", "Gs5", "A5", "As5", "B5", "C6", "Cs6", "D6", "Ds6", "E6", "F6", "Fs6", "G6", "Gs6", "A6", "As6", "B6", "C7", "Cs7", "D7", "Ds7", "E7", "F7", "Fs7", "G7", "Gs7", "A7", "As7", "B7", "C8", "Cs8", "D8", "Ds8", "E8", "F8", "Fs8", "G8", "Gs8", "A8", "As8", "B8", "C9", "Cs9", "D9", "Ds9", "E9", "F9", "Fs9", "G9"]

def midifile_df_create():
    
    # グローバル変数
    global midifile_path
    
    # 保存先ディレクトリ
    datas_dir = "datas"
    os.makedirs(datas_dir, exist_ok=True)

    # ファイルパスの確認
    if 'midifile_path' not in globals():
        print("エラー: midifile_path が定義されていません。先に load_midi() を実行してください。")
        return

    file_name_base = str(os.path.splitext(os.path.basename(midifile_path))[0])
    dat_output_path = os.path.join(datas_dir, f"{file_name_base}.dat")
    
    # ファイル名重複回避
    counter = 1
    while os.path.exists(dat_output_path):
        dat_output_path = os.path.join(datas_dir, f"{file_name_base} ({counter}).dat")
        counter += 1

    dat_content_lines = [] 
    
    ##########################################################################
    #######   MIDI読み込みと基本情報の取得   ########
    ##########################################################################

    try:
        midifile = MidiFile(midifile_path)
    except Exception as e:
        print(f"エラー: MIDIファイルの読み込みに失敗しました: {e}")
        return
    
    line = "// File name: " + file_name_base
    print(line); dat_content_lines.append(line)
    
    num_midi_tracks = len(midifile.tracks)
    line = "// Number of track in the MIDI file: " + str(num_midi_tracks)
    print(line); dat_content_lines.append(line)
    
    ticks_per_beat = midifile.ticks_per_beat
    line = "// Ticks per beat of the MIDI file: " + str(ticks_per_beat)
    print(line); dat_content_lines.append(line)
    
    # --- BPM と 拍子の取得 ---
    tempo_found = False
    bpm = 120 
    tempo = 500000 
    numerator = 4 
    denominator = 4 
    time_sig_found = False

    for track in midifile.tracks:
        for msg in track:
            if msg.is_meta:
                if msg.type == 'set_tempo':
                    tempo = msg.tempo
                    bpm = int(round(mido.tempo2bpm(tempo),2))
                    tempo_found = True
                elif msg.type == 'time_signature':
                    numerator = msg.numerator
                    denominator =  msg.denominator
                    time_sig_found = True
    
    if not tempo_found:
        print(f"警告: BPM情報なし。デフォルト {bpm} を使用。")
    if not time_sig_found:
        print(f"警告: 拍子情報なし。デフォルト {numerator}/{denominator} を使用。")

    # --- BPMの確認・上書き ---
    print("--- BPM確認 ---")
    bpm_input = input(f"検出されたBPM: {bpm} (このままEnter, または新しいBPMを入力) >> ")
    if bpm_input.strip(): 
        try:
            new_bpm = int(bpm_input.strip())
            print(f"BPMを {new_bpm} に上書きします。")
            bpm = new_bpm
            tempo = mido.bpm2tempo(bpm) 
        except ValueError:
            print(f"無効な入力です。元のBPM {bpm} を使用します。")
    
    line = f"// Tempo of the MIDI file: {tempo}"
    dat_content_lines.append(line)
    line = f"// BPM: {bpm}"
    print(line); dat_content_lines.append(line)
    dat_content_lines.append(f"// Time signature: {numerator}/{denominator}")

    ##########################################################################
    #######   ノートデータの解析 (ms単位への変換とリスト化)   ########
    ##########################################################################
    
    # トラック選択 (最初のNoteOnがあるトラックを探す)
    target_track_index = -1 
    for idx, track in enumerate(midifile.tracks):
        for msg in track:
            if msg.type == 'note_on' and msg.velocity > 0:
                target_track_index = idx 
                break
        if target_track_index != -1: break 

    if target_track_index == -1:
        print("エラー: ノートが見つかりません。処理を中断します。")
        return 
    
    print(f"// Note: MIDIトラック {target_track_index + 1} を解析します。")
    dat_content_lines.append(f"// Note: MIDI Track {target_track_index + 1}")

    # --- 1. ノートの生データを抽出 (Tick単位) ---
    raw_notes = [] # {'note': 60, 'start_tick': 0, 'end_tick': 480}
    curr_tick = 0
    active_notes = {} # key=note_num, val=start_tick

    for msg in midifile.tracks[target_track_index]:
        curr_tick += msg.time
        
        if msg.type == 'note_on' and msg.velocity > 0:
            active_notes[msg.note] = curr_tick
        elif (msg.type == 'note_off') or (msg.type == 'note_on' and msg.velocity == 0):
            if msg.note in active_notes:
                start_t = active_notes.pop(msg.note)
                end_t = curr_tick
                raw_notes.append({
                    'note': msg.note,
                    'start_tick': start_t,
                    'end_tick': end_t
                })
    
    # 時間順にソート
    raw_notes.sort(key=lambda x: x['start_tick'])

    if not raw_notes:
        print("エラー: 有効なノートが抽出できませんでした。")
        return

    # --- 2. ms(ミリ秒)への変換 ---
    ms_per_tick = tempo / ticks_per_beat / 1000.0
    
    for n in raw_notes:
        n['start_ms'] = int(n['start_tick'] * ms_per_tick)
        n['end_ms']   = int(n['end_tick'] * ms_per_tick)

    ##########################################################################
    #######   休符 (Rest) の挿入ロジック   ########
    ##########################################################################
    
    final_notes = []      # 整形後のノート番号リスト
    final_avail_time = [] # 整形後のavailableTimeリスト
    
    # 設定パラメータ (ms)
    REST_THRESHOLD_MS = 100  # これ以上の隙間があれば休符を入れる
    MIN_TRANSITION_MS = 30   # 音の切り替えに必要な最小時間
    
    # 最初のノート (スタート準備)
    final_notes.append(raw_notes[0]['note'])
    final_avail_time.append(1000) # 最初の音は十分な準備時間を与える

    for i in range(len(raw_notes) - 1):
        curr_n = raw_notes[i]
        next_n = raw_notes[i+1]
        
        # 隙間 (Gap) を計算
        gap_ms = next_n['start_ms'] - curr_n['end_ms']
        
        if gap_ms > REST_THRESHOLD_MS:
            # --- パターンA: 隙間が大きい -> 休符を入れる ---
            
            # 1. 現在の音 -> 休符への切り替え
            final_avail_time.append(MIN_TRANSITION_MS)
            
            # 2. 休符イベントの追加 (MIDI番号 0)
            final_notes.append(0) 
            
            # 3. 休符 -> 次の音への移動時間 (実際のギャップ時間)
            final_avail_time.append(gap_ms)
            
            # 4. 次の音符の追加
            final_notes.append(next_n['note'])
            
        else:
            # --- パターンB: 隙間が小さい (Legato / Slide) ---
            
            # 物理的に移動可能な時間は max(gap, 最小値)
            transition_time = max(gap_ms, MIN_TRANSITION_MS)
            
            final_avail_time.append(transition_time)
            final_notes.append(next_n['note'])

    # ★修正: 最後の要素追加を削除し、配列長を一致させる
    # 以前のコードではここで final_avail_time.append(1000) をしていたため
    # availableTime が note より1つ多くなりエラーになっていました。
    
    # 安全策: 長さが一致しているか確認
    if len(final_notes) != len(final_avail_time):
        print(f"警告: 配列長不一致 (Note: {len(final_notes)}, Time: {len(final_avail_time)})。調整します。")
        # 少ない方に合わせるか、多い方を切り詰める（基本は一致するはず）
        min_len = min(len(final_notes), len(final_avail_time))
        final_notes = final_notes[:min_len]
        final_avail_time = final_avail_time[:min_len]

    num_final_notes = len(final_notes)
    line = f"// Number of notes (with Rests): {num_final_notes}"
    print(line); dat_content_lines.append(line)

    ##########################################################################
    #######   楽器データの自動生成 (Integer Base)   ########
    ##########################################################################
    
    # MIDI番号 40(E2) ～ 88(E6) 程度をカバー
    midi_range = range(40, 89)
    
    # PlayableSounds: {0, 40, 41, ...} (0は休符)
    playable_sounds_str = "0," + ",".join(str(n) for n in midi_range)
    
    # 定義: {MIDI番号: 弦}
    open_strings_map = {40:6, 45:5, 50:4, 55:3, 59:2, 64:1}
    # 定義: {弦: 開放弦のMIDI番号}
    string_bases = {6:40, 5:45, 4:50, 3:55, 2:59, 1:64}
    
    open_strings_list = []
    combinations_list = []
    f_max = 22
    
    # --- 休符(0)の定義 ---
    open_strings_list.append("{}") # 0番目
    combinations_list.append("{}") # 0番目
    
    # --- 各ノートの定義 ---
    for note_num in midi_range:
        # OpenStrings
        if note_num in open_strings_map:
            open_strings_list.append(f"{{{open_strings_map[note_num]}}}")
        else:
            open_strings_list.append("{}")
            
        # Combinations
        combs = []
        for string_idx in range(1, 7):
            base = string_bases[string_idx]
            fret = note_num - base
            if 0 <= fret <= f_max:
                combs.append(f"<{string_idx},{fret}>")
        
        if not combs:
            combinations_list.append("{}")
        else:
            combinations_list.append("{" + ",".join(combs) + "}")
            
    open_str = ",".join(open_strings_list)
    comb_str = ",".join(combinations_list)
    
    # 音名配列 (文字列)
    all_names_str = ",".join(f'"{name}"' for name in notes_names)

    instrument_section = f"""
//--------------------------instrument (Auto Generated / Integer)
f_max = {f_max};
s_max = 6;
PlayableSounds = {{{playable_sounds_str}}};
OpenStrings = [{open_str}];
Combinations = [{comb_str}];

// Display Name Mapping
MidiNames = [{all_names_str}]; 
"""
    dat_content_lines.append(instrument_section)

    ##########################################################################
    #######   Melody & Config 出力   ########
    ##########################################################################
    
    notes_str = ",".join(str(n) for n in final_notes)
    times_str = ",".join(str(t) for t in final_avail_time)
    
    melody_section = f"""
//---------------------------melody
m = {num_final_notes};
note = [{notes_str}]; // MIDI Numbers (0=Rest)
availableTime = [{times_str}];
"""
    
    print(melody_section)
    dat_content_lines.append(melody_section)

    optimization_settings = ""
    try:
        with open('optimization_settings.txt', 'r') as opt:   
            optimization_settings = opt.read()
    except FileNotFoundError:
        line = "警告: 'optimization_settings.txt' が見つかりません。"
        print(line); dat_content_lines.append(f"// {line}")
        optimization_settings = "// 'optimization_settings.txt' が見つからなかったため、デフォルト設定を書き込めません。"

    customize_request = input("//Do you want to customize the configuration? Answer Y or N ")    
    if customize_request.upper() == "Y":
        print("\\n--- 物理パラメータ(mm)の入力 ---")
        scaleLength = input("// 1. スケール長 (mm) (例: 635.0) >> ")
        stringSpacing = input("// 2. 弦の間隔 (mm) (例: 10.8) >> ")
        SP_max_mm = input("// 3. 指の最大広がり (mm) [中,薬,小指] (例: [40.0, 50.0, 60.0]) >> ")
        timePerMm = input("// 4. 1mmあたりの移動時間 (ms) (例: 0.5) >> ")
        fingerLengths_mm = input("// 5. 各指の長さ (mm) [人,中,薬,小] (例: [80.0, 85.0, 83.0, 75.0]) >> ")
        pc_threshold_mm = input("// 6. PCコストが固定になる閾値 (mm) (例: 50.0) >> ")
        
        print("\\n--- ペナルティ係数 (d_X) の入力 ---")
        d_PC = input("// 7. d_PC (ポジション変更) (例: 1.0) >> ")
        d_SC = input("// 8. d_SC (弦変更) (例: 1.0) >> ")
        d_HS = input("// 9. d_HS (手の広がり) (例: 2.0) >> ")
        d_DH = input("// 10. d_DH (到達コスト) (例: 0.5) >> ")
        d_NO = input("// 11. d_NO (開放弦) (例: 100.0) >> ")

    else:
        print(optimization_settings) 
        dat_content_lines.append(optimization_settings)
    
    # 書き込み
    final_content = "\n".join(dat_content_lines)
    try:
        with open(dat_output_path, 'w', encoding='utf-8') as f:
            f.write(final_content)
        print(f"\n--- 成功 ---")
        print(f".dat ファイルが {dat_output_path} に保存されました。")
    except Exception as e:
        print(f"\n--- エラー ---")
        print(f".dat ファイルの保存に失敗しました: {e}")

# --- 関数を実行 ---
midifile_df_create()

// File name: complex_solo
// Number of track in the MIDI file: 2
// Ticks per beat of the MIDI file: 480
--- BPM確認 ---


検出されたBPM: 190 (このままEnter, または新しいBPMを入力) >>  


// BPM: 190
// Note: MIDIトラック 1 を解析します。
// Number of notes (with Rests): 111

//---------------------------melody
m = 111;
note = [69,69,69,67,65,64,65,64,62,61,70,61,68,69,74,76,77,76,77,79,77,79,81,0,84,84,0,82,81,79,77,0,76,77,76,74,73,72,81,72,70,79,70,69,77,76,74,72,0,69,70,72,77,79,81,79,77,69,70,69,67,65,64,62,61,0,57,62,60,62,65,65,65,65,67,67,67,67,69,69,72,72,74,74,76,76,77,77,77,77,79,79,79,79,81,84,73,77,84,82,77,73,72,73,72,70,73,0,70,0,69]; // MIDI Numbers (0=Rest)
availableTime = [1000,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,474,30,30,159,30,30,30,30,159,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,317,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,238,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,158,30,158];



//Do you want to customize the configuration? Answer Y or N  N


//-------------------------configuration
connectedTimeThreshold = 10.0; 
rest_pc_discount = 0.1;
M_value = 800.0;
scaleLength = 635.0;
stringSpacing = 10.8;
SP_max_mm = [60.0,80.0,100.0];
natural_min_spread_mm = [0, 0, 0];
timePerMm = 0.5;

fingerLengths_mm = [70.0, 75.0, 70.0, 50.0];
// Finger difficulty multiplier (1.0=easiest, 1.5=mid, 2.0=ring, 2.5=pinky)
fingerDifficulty = [1.0, 1.1, 1.6, 2.2];
timeThresholdRatio = 1.5;
highPositionPenalty = 2.0;

d_PC = 1.2; 
d_SC = 1.0;  
d_HS = 0.8;
d_DH = 2.0;
d_NO = 5.0; 
              
              
// --- (EXISTING) alpha（重み） ---
alpha_PC = 0.2; //position change
alpha_SC = 0.2; //string change
alpha_HS = 0.2; //hand spread
alpha_DH = 0.2; //distance from the handy position
alpha_NO = 0.2; //number of notes played on open strings

--- 成功 ---
.dat ファイルが datas/complex_solo (1).dat に保存されました。
